# AI Drama Production Studio — RunPod Launch

Run each cell **top to bottom** on first use. After that, only Cell 4 (launch) is needed on restart.

**Pod recommendation:** RTX Pro 6000 (96 GB VRAM) · 86 GB RAM · 500 GB storage

In [ ]:
# Cell 1 — Verify GPU
import subprocess, sys
print(subprocess.check_output('nvidia-smi', shell=True).decode())
print('Python:', sys.version)

In [ ]:
# Cell 2 — Install dependencies
# PyTorch CUDA 12.1 wheel (matches RunPod base images)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

# Core ML + app dependencies
!pip install -r /workspace/AI-Studio-Producer/studio/requirements.txt -q

# Verify torch+CUDA
import torch
print(f'Torch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Cell 3 — Install ffmpeg (required for video export and episode assembly)
!apt-get install -y ffmpeg 2>/dev/null | tail -5
!ffmpeg -version | head -1

In [ ]:
# Cell 4 — Launch the studio
# The public Gradio URL is printed below — open it in your browser.
# On RunPod you can also use the HTTP port proxy on port 7860.
import subprocess, os, sys, time

STUDIO_PATH = '/workspace/AI-Studio-Producer'
STUDIO_ENV  = 'production'   # uses /workspace/* paths for logs/output
PORT        = 7860

if STUDIO_PATH not in sys.path:
    sys.path.insert(0, STUDIO_PATH)

env = os.environ.copy()
env['STUDIO_ENV']  = STUDIO_ENV
env['STUDIO_ROOT'] = STUDIO_PATH + '/studio'
env['PORT']        = str(PORT)

proc = subprocess.Popen(
    [sys.executable, f'{STUDIO_PATH}/studio/app.py', '--port', str(PORT), '--share'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print('Studio starting — watching logs for Gradio URL...')
for line in proc.stdout:
    print(line, end='')
    if 'Running on' in line or 'gradio.live' in line:
        print('\n✅ Studio is live. Copy the URL above.')
        break

In [ ]:
# Cell 5 — Stream logs (run in a separate cell while the studio is live)
# Interrupt kernel to stop streaming.
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    print('Log stream stopped.')

In [ ]:
# Cell 6 — Stop the studio
proc.terminate()
print('Studio stopped.')

## Model paths (set once in the UI, saved to DB)

After launch, open the **Video Generation** tab → **Video Model Manager** and set:

| Model | Path on RunPod |
|---|---|
| LTX-Video 2.x base | `/workspace/models/ltx-video-2b-v0.9.7-distilled` |
| LTX Upscaler | `/workspace/models/ltx-video-2b-v0.9.7-distilled-04-25-upscaler` |
| Wan 2.2 T2V | `/workspace/models/Wan2.2-T2V-14B` |
| Hunyuan Video | `/workspace/models/HunyuanVideo` |
| LoRA directory | `/workspace/loras/` |

Click **Save Paths** then **Load Pipeline** before queuing any generation jobs.

## Download models (first time only)

```bash
# LTX-Video (recommended starting model — fastest on this pod)
huggingface-cli download Lightricks/LTX-Video-0.9.7-distilled \
    --local-dir /workspace/models/ltx-video-2b-v0.9.7-distilled

# Wan 2.2 T2V 14B
huggingface-cli download Wan-AI/Wan2.2-T2V-14B \
    --local-dir /workspace/models/Wan2.2-T2V-14B

# Hunyuan Video
huggingface-cli download tencent/HunyuanVideo \
    --local-dir /workspace/models/HunyuanVideo
```

## Lip sync engines (optional)

```bash
# Wav2Lip (most reliable)
git clone https://github.com/Rudrabha/Wav2Lip /workspace/Wav2Lip
# Download checkpoint to /workspace/Wav2Lip/checkpoints/wav2lip_gan.pth
# Then set lipsync_wav2lip_code_dir in Settings tab
```